# FILE 1F: CENTRALIZED TRAINING -- MuRIL (RETRAIN)
## Replaces your lost checkpoint/results file
### Same setup as File 1 (Dropout=0.4, LR=2e-5, Early Stopping) + fp32 fix, 10 epochs

**Estimated time: ~25-30 min (10 epochs) on T4 x2**

**Dataset paths already filled in** (same Kaggle dataset as your DistilXLM-R run: `foysal2004042/againth`) -- no need to edit Cell 2.

**After disconnect:** Run Cell 1 only, then continue from where you stopped.

**This time, download `results_muril.json` immediately after Cell 4 finishes and keep a backup** -- this is the exact file you lost last time.

## ⚡ Cell 1: COMPLETE SETUP -- Run after any disconnect

In [8]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, copy, time, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    matthews_corrcoef, confusion_matrix
)
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader

CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_MULTI_GPU = torch.cuda.device_count() > 1

class DeceptionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts  = df['text'].fillna('').tolist()
        self.labels = df['is_deceptive'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

class AttentionDeceptionClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.4):
        super().__init__()
        # NOTE: some HF checkpoints ship fp16 weights by default, which raises
        # "mat1 and mat2 must have the same dtype, but got Half and Float" once
        # combined with this float32 classifier head. Forcing float32 at load
        # time avoids that mismatch for every backbone (see thesis Section 3.3.5,
        # "Precision Handling").
        self.encoder = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32).float()
        h = self.encoder.config.hidden_size
        self.attention_pool = nn.Sequential(
            nn.Linear(h, 256), nn.Tanh(), nn.Linear(256, 1)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(h, 256), nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, 2)
        )
    def forward(self, input_ids, attention_mask):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        scores = self.attention_pool(hidden)
        scores = scores.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)
        pooled = (hidden * torch.softmax(scores, dim=1)).sum(dim=1)
        return self.classifier(pooled)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['label'].to(device)
            logits = model(ids, mask)
            loss   = criterion(logits, lbls)
            probs  = torch.softmax(logits, dim=1)[:, 1]
            preds  = logits.argmax(dim=1)
            total_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return {
        'loss':      total_loss / len(loader),
        'acc':       accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall':    recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1':        f1_score(all_labels, all_preds, average='macro'),
        'f1_w':      f1_score(all_labels, all_preds, average='weighted'),
        'auc':       roc_auc_score(all_labels, all_probs),
        'mcc':       matthews_corrcoef(all_labels, all_preds),
        'preds':     all_preds,
        'labels':    all_labels,
    }

def train_model(model_name, label, epochs=10, batch_size=32, lr=2e-5, patience=2):
    ckpt_path    = f'{CKPT_DIR}/{label}_best.pt'
    results_path = f'{CKPT_DIR}/{label}_results.json'

    if os.path.exists(results_path):
        print(f'\n  checkmark  {label} already trained -- loading saved results (no retraining)')
        with open(results_path) as f:
            return json.load(f)

    print(f'\n' + '='*70)
    print(f'TRAINING: {label}')
    print(f'='*70)
    print(f'  Model:         {model_name}')
    print(f'  Max Epochs:    {epochs} (early stopping patience={patience})')
    print(f'  LR:            {lr}')
    print(f'  Dropout:       0.4')
    print(f'  Weight Decay:  0.02')

    tokenizer    = AutoTokenizer.from_pretrained(model_name)
    train_loader = DataLoader(DeceptionDataset(train_df, tokenizer), batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(DeceptionDataset(val_df,   tokenizer), batch_size=batch_size*2, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(DeceptionDataset(test_df,  tokenizer), batch_size=batch_size*2, shuffle=False, num_workers=2, pin_memory=True)

    model = AttentionDeceptionClassifier(model_name, dropout=0.4)
    if USE_MULTI_GPU:
        model = nn.DataParallel(model)
    model = model.to(DEVICE)

    total_steps  = len(train_loader) * epochs
    optimizer    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.02)
    scheduler    = get_linear_schedule_with_warmup(optimizer, int(total_steps*0.1), total_steps)
    criterion    = nn.CrossEntropyLoss()

    best_f1    = 0
    best_state = None
    no_improve = 0
    history    = []
    t0         = time.time()

    for epoch in range(1, epochs + 1):
        print(f'\n  [EPOCH {epoch}/{epochs}]')
        print('  ' + '-'*66)
        model.train()
        train_loss = 0

        for idx, batch in enumerate(train_loader):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(ids, mask), lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
            if (idx+1) % 50 == 0 or (idx+1) == len(train_loader):
                print(f'    Batch {idx+1}/{len(train_loader)} | Loss: {train_loss/(idx+1):.4f}')

        avg_loss = train_loss / len(train_loader)
        val_m    = evaluate(model, val_loader, criterion, DEVICE)

        print(f'\n  EPOCH {epoch} RESULTS:')
        print(f'    Train Loss: {avg_loss:.4f}')
        print(f'    Val Loss:   {val_m["loss"]:.4f}')
        print(f'    Val Acc:    {val_m["acc"]:.4f}')
        print(f'    Val F1:     {val_m["f1"]:.4f}')
        print(f'    Val AUC:    {val_m["auc"]:.4f}')
        print(f'    Val MCC:    {val_m["mcc"]:.4f}')

        history.append({
            'epoch': epoch, 'train_loss': avg_loss,
            'val_loss': val_m['loss'], 'val_acc': val_m['acc'],
            'val_f1': val_m['f1'], 'val_auc': val_m['auc'], 'val_mcc': val_m['mcc']
        })

        if val_m['f1'] > best_f1:
            best_f1    = val_m['f1']
            best_state = copy.deepcopy(
                model.module.state_dict() if USE_MULTI_GPU else model.state_dict()
            )
            torch.save(best_state, ckpt_path)
            no_improve = 0
            print(f'    NEW BEST saved (F1={best_f1:.4f})')
        else:
            no_improve += 1
            print(f'    No improvement ({no_improve}/{patience})')
            if no_improve >= patience:
                print(f'\n  EARLY STOPPING at epoch {epoch}')
                break

    if USE_MULTI_GPU:
        model.module.load_state_dict(torch.load(ckpt_path))
    else:
        model.load_state_dict(torch.load(ckpt_path))

    test_m  = evaluate(model, test_loader, criterion, DEVICE)
    elapsed = (time.time() - t0) / 60

    print(f'\n  Training time: {elapsed:.1f} min')
    print(f'  FINAL TEST RESULTS:')
    print(f'    Accuracy:      {test_m["acc"]:.4f}')
    print(f'    Precision:     {test_m["precision"]:.4f}')
    print(f'    Recall:        {test_m["recall"]:.4f}')
    print(f'    F1 (Macro):    {test_m["f1"]:.4f}  <- MAIN METRIC')
    print(f'    F1 (Weighted): {test_m["f1_w"]:.4f}')
    print(f'    AUC-ROC:       {test_m["auc"]:.4f}')
    print(f'    MCC:           {test_m["mcc"]:.4f}')

    results = {
        'model': label,
        'accuracy': float(test_m['acc']),
        'precision': float(test_m['precision']),
        'recall': float(test_m['recall']),
        'macro_f1': float(test_m['f1']),
        'weighted_f1': float(test_m['f1_w']),
        'auc_roc': float(test_m['auc']),
        'mcc': float(test_m['mcc']),
        'history': history,
        'train_minutes': elapsed,
        'preds': [int(x) for x in test_m['preds']],
        'labels': [int(x) for x in test_m['labels']],
    }
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    del model, tokenizer, train_loader, val_loader, test_loader
    torch.cuda.empty_cache()
    return results

print('='*70)
print('SETUP COMPLETE - ALL FUNCTIONS READY')
print('='*70)
print(f'Device:     {DEVICE}')
print(f'Multi-GPU:  {USE_MULTI_GPU}')
print(f'Checkpoint: {CKPT_DIR}')
print()
print('IF NETWORK DISCONNECTED:')
print('  1. Re-run ONLY this cell (Cell 1)')
print('  2. Re-run Cell 2 (load data)')
print('  3. Continue from the training cell')
print('  4. Already-trained model will be SKIPPED automatically')
print('='*70)

SETUP COMPLETE - ALL FUNCTIONS READY
Device:     cuda
Multi-GPU:  True
Checkpoint: /kaggle/working/checkpoints

IF NETWORK DISCONNECTED:
  1. Re-run ONLY this cell (Cell 1)
  2. Re-run Cell 2 (load data)
  3. Continue from the training cell
  4. Already-trained model will be SKIPPED automatically


## Cell 2: Load Data (re-run after disconnect)
Uses the exact same `random_state=42` split as File 1 and your DistilXLM-R notebook -- test set will be identical (2,601 rows), so results stay directly comparable and the per-domain breakdown code will work unchanged.

In [9]:
task_files = {
    'fake_news':            '/kaggle/input/datasets/foysal2004042/againth/fake_news_sampled_multilingual.csv',
    'product_reviews':      '/kaggle/input/datasets/foysal2004042/againth/product_reviews_sampled_multilingual.csv',
    'phishing':             '/kaggle/input/datasets/foysal2004042/againth/phishing_sampled_multilingual.csv',
    'political_statements': '/kaggle/input/datasets/foysal2004042/againth/political_statements_sampled_multilingual.csv',
    'job_scams':            '/kaggle/input/datasets/foysal2004042/againth/job_scams_sampled_multilingual.csv',
}

print('Loading data...')
dfs = []
for task, path in task_files.items():
    try:
        df = pd.read_csv(path)
        df['task'] = task
        dfs.append(df)
        print(f'  ok {task:<25} {len(df):>6,} rows')
    except FileNotFoundError:
        print(f'  FAIL {task} NOT FOUND')

df_all = pd.concat(dfs, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
train_df, temp_df = train_test_split(df_all, test_size=0.30, random_state=42, stratify=df_all['is_deceptive'])
val_df, test_df   = train_test_split(temp_df, test_size=0.6667, random_state=42, stratify=temp_df['is_deceptive'])

print(f'\nTrain: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print('Data ready -- same 42-seeded split as File 1 / DistilXLM-R notebook, so results stay comparable')

Loading data...
  ok fake_news                  6,496 rows
  ok product_reviews            2,238 rows
  ok phishing                   1,654 rows
  ok political_statements       1,394 rows
  ok job_scams                  1,216 rows

Train: 9,098 | Val: 1,299 | Test: 2,601
Data ready -- same 42-seeded split as File 1 / DistilXLM-R notebook, so results stay comparable


## Cell 3: Train MuRIL (~25-30 min (10 epochs), 10 epochs max)

In [10]:
results_muril = train_model(
    model_name = 'google/muril-base-cased',
    label      = 'MuRIL',
    epochs     = 10,
    batch_size = 32,
    lr         = 2e-5,
    patience   = 2
)
print('MuRIL F1:', results_muril['macro_f1'])


TRAINING: MuRIL
  Model:         google/muril-base-cased
  Max Epochs:    10 (early stopping patience=2)
  LR:            2e-05
  Dropout:       0.4
  Weight Decay:  0.02


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]


  [EPOCH 1/10]
  ------------------------------------------------------------------
    Batch 50/285 | Loss: 0.6938
    Batch 100/285 | Loss: 0.6944
    Batch 150/285 | Loss: 0.6937
    Batch 200/285 | Loss: 0.6894
    Batch 250/285 | Loss: 0.6827
    Batch 285/285 | Loss: 0.6777

  EPOCH 1 RESULTS:
    Train Loss: 0.6777
    Val Loss:   0.6333
    Val Acc:    0.7229
    Val F1:     0.7228
    Val AUC:    0.7978
    Val MCC:    0.4460
    NEW BEST saved (F1=0.7228)

  [EPOCH 2/10]
  ------------------------------------------------------------------
    Batch 50/285 | Loss: 0.6289
    Batch 100/285 | Loss: 0.6156
    Batch 150/285 | Loss: 0.6023
    Batch 200/285 | Loss: 0.5907
    Batch 250/285 | Loss: 0.5833
    Batch 285/285 | Loss: 0.5757

  EPOCH 2 RESULTS:
    Train Loss: 0.5757
    Val Loss:   0.5306
    Val Acc:    0.7691
    Val F1:     0.7689
    Val AUC:    0.8586
    Val MCC:    0.5387
    NEW BEST saved (F1=0.7689)

  [EPOCH 3/10]
  ----------------------------------------

## Cell 4: Save Result -- DOWNLOAD THIS FILE IMMEDIATELY

In [ ]:
# Load from checkpoint (safe even after disconnect)
results_muril = json.load(open(f'{CKPT_DIR}/MuRIL_results.json'))

summary = pd.DataFrame([
    {'Model': 'MuRIL', 'Learning Mode': 'Centralized', 'Best Round': '-',
      'Accuracy': results_muril['accuracy'], 'Precision': results_muril['precision'],
      'Recall': results_muril['recall'], 'F1-Macro': results_muril['macro_f1'],
      'AUC-ROC': results_muril['auc_roc'], 'MCC': results_muril['mcc']},
])

pd.set_option('display.float_format', '{:.4f}'.format)
print('=' * 70)
print('MuRIL CENTRALIZED RESULT (retrained -- replaces lost checkpoint)')
print('=' * 70)
print(summary.to_string(index=False))

with open('/kaggle/working/results_muril.json', 'w') as f:
    json.dump({'MuRIL': results_muril}, f, indent=2)
summary.to_csv('/kaggle/working/centralized_muril_result.csv', index=False)

print('\nSaved: results_muril.json and centralized_muril_result.csv')
print('\nIMPORTANT: download BOTH files this time and keep a backup copy --')
print('this is the file you lost last time. It also contains full per-example')
print('preds/labels, needed for the per-domain and per-language breakdown.')
print('\nNEXT STEPS:')
print('  1. Download results_muril.json and centralized_muril_result.csv')
print('  2. Re-append this row to your All_Result_Merge.xlsx (replacing the lost one)')
print('  3. Re-run the per-domain/per-language breakdown for MuRIL, same as DistilXLM-R')

## Cell 5: Training Curve

In [ ]:
res = json.load(open(f'{CKPT_DIR}/MuRIL_results.json'))

fig, ax = plt.subplots(1, 1, figsize=(6.5, 5))
ax2 = ax.twinx()

epochs_list = [h['epoch']      for h in res['history']]
f1_list     = [h['val_f1']     for h in res['history']]
tl_list     = [h['train_loss'] for h in res['history']]
vl_list     = [h['val_loss']   for h in res['history']]

ax.plot(epochs_list, f1_list, 'o-', color='#185FA5', linewidth=2.5, markersize=8, label='Val F1')
ax2.plot(epochs_list, tl_list, 's--', color='gray',   linewidth=1.5, label='Train Loss')
ax2.plot(epochs_list, vl_list, 'd--', color='orange', linewidth=1.5, label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Val F1', color='#185FA5')
ax2.set_ylabel('Loss')
ax.set_title(f'MuRIL\nTest F1={res["macro_f1"]:.4f}', fontweight='bold')
ax.grid(True, alpha=0.3)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.suptitle('MuRIL -- Centralized Training Curve (retrained)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/muril_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: muril_training_curve.png')